In [1]:
import pandas as pd
train = pd.read_csv('/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/train.csv')
test = pd.read_csv('/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/test.csv')
train_labels = pd.read_csv('/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/train_labels.csv')
target_pairs = pd.read_csv('/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/target_pairs.csv')

In [2]:
import re

def parse_pair(pair_str):
    """Returns list of (sign, ticker) tuples. Sign is +1 or -1."""
    # Normalize whitespace
    s = pair_str.strip()
    # Split on + or - while keeping the operator
    tokens = re.split(r'\s*([+-])\s*', s)
    # tokens looks like ['LME_AH_Close', '-', 'LME_ZS_Close'] or just ['LME_AH_Close']
    result = []
    sign = 1
    if tokens[0]:
        result.append((sign, tokens[0]))
    i = 1
    while i < len(tokens):
        op = tokens[i]
        ticker = tokens[i+1]
        sign = 1 if op == '+' else -1
        result.append((sign, ticker))
        i += 2
    return result

def get_exchange(ticker):
    """Pull the prefix that identifies the venue/family."""
    # Typical prefixes in this dataset: LME_, JPX_, US_Stock_, FX_
    if ticker.startswith('US_Stock_'):
        return 'US_Stock'
    return ticker.split('_')[0]

def get_instrument(ticker):
    """Strip exchange prefix and _Close/_adj_close suffix to get the symbol."""
    t = ticker
    for prefix in ['US_Stock_', 'LME_', 'JPX_', 'FX_']:
        if t.startswith(prefix):
            t = t[len(prefix):]
            break
    for suffix in ['_Close', '_adj_close', '_close']:
        if t.endswith(suffix):
            t = t[:-len(suffix)]
            break
    return t

In [3]:
rows = []
for _, r in target_pairs.iterrows():
    components = parse_pair(r['pair'])
    tickers = [t for _, t in components]
    exchanges = sorted(set(get_exchange(t) for t in tickers))
    instruments = [get_instrument(t) for t in tickers]
    rows.append({
        'target': r['target'],
        'lag': r['lag'],
        'pair_raw': r['pair'],
        'n_legs': len(components),
        'is_spread': len(components) > 1,
        'tickers': tickers,
        'instruments': instruments,
        'exchanges': exchanges,
        'exchange_key': '_'.join(exchanges),  # e.g. 'LME', 'LME_JPX'
    })

target_info = pd.DataFrame(rows)
print(target_info.head())

     target  lag                                        pair_raw  n_legs  \
0  target_0    1                           US_Stock_VT_adj_close       1   
1  target_1    1            LME_PB_Close - US_Stock_VT_adj_close       2   
2  target_2    1                     LME_CA_Close - LME_ZS_Close       2   
3  target_3    1                     LME_AH_Close - LME_ZS_Close       2   
4  target_4    1  LME_AH_Close - JPX_Gold_Standard_Futures_Close       2   

   is_spread                                          tickers  \
0      False                          [US_Stock_VT_adj_close]   
1       True            [LME_PB_Close, US_Stock_VT_adj_close]   
2       True                     [LME_CA_Close, LME_ZS_Close]   
3       True                     [LME_AH_Close, LME_ZS_Close]   
4       True  [LME_AH_Close, JPX_Gold_Standard_Futures_Close]   

                   instruments        exchanges  exchange_key  
0                         [VT]       [US_Stock]      US_Stock  
1                     [P

In [4]:
# How many targets per exchange grouping
print(target_info.groupby('exchange_key').size().sort_values(ascending=False))

# How many targets per lag
print(target_info.groupby('lag').size())

# Single-leg vs spread breakdown
print(target_info.groupby(['is_spread', 'exchange_key']).size())

# Most common instruments across targets
from collections import Counter
all_instruments = Counter()
for insts in target_info['instruments']:
    all_instruments.update(insts)
print(all_instruments.most_common(20))

exchange_key
LME_US_Stock    181
JPX_US_Stock     87
FX_LME           86
FX_JPX           46
JPX_LME          12
LME               8
FX                2
US_Stock          2
dtype: int64
lag
1    106
2    106
3    106
4    106
dtype: int64
is_spread  exchange_key
False      FX                2
           US_Stock          2
True       FX_JPX           46
           FX_LME           86
           JPX_LME          12
           JPX_US_Stock     87
           LME               8
           LME_US_Stock    181
dtype: int64
[('AH', 81), ('ZS', 76), ('Gold_Standard_Futures', 74), ('Platinum_Standard_Futures', 71), ('CA', 70), ('PB', 68), ('VT', 5), ('ZARUSD', 5), ('NOKEUR', 5), ('VXUS', 5), ('VYM', 4), ('IEMG', 4), ('AUDJPY', 4), ('VGIT', 4), ('EURAUD', 4), ('CADCHF', 4), ('ZARCHF', 4), ('NZDJPY', 4), ('CHFJPY', 4), ('VTV', 4)]


In [6]:
def features_for_target(target_name, target_info, feature_cols, scope='family'):
    """
    scope='direct'  : just the tickers in the target's pair
    scope='family'  : direct + all features from same exchange(s)
    scope='all'     : everything (fall back when family is too narrow)
    """
    row = target_info[target_info['target'] == target_name].iloc[0]
    direct = [t for t in row['tickers'] if t in feature_cols]
    if scope == 'direct':
        return direct
    family_cols = [c for c in feature_cols
                   if get_exchange(c) in row['exchanges']]
    if scope == 'family':
        return sorted(set(direct) | set(family_cols))
    return list(feature_cols)

feature_cols = [c for c in train.columns if c != 'date_id']

# Quick sanity check
print(features_for_target('target_0', target_info, feature_cols, scope='direct'))
print(len(features_for_target('target_0', target_info, feature_cols, scope='family')))

['US_Stock_VT_adj_close']
475
